# How To: BCI Motor Imagery (MI) offline pipeline


## Step 1: paradigm

In this step we want to show how to define a paradigm for MI applications to obtain the needed signals. 
Normally written in *Psychopy*. 


Vitalik pls erklär wie des geht danke xoxo :3


## Step 2: preprocessing

Once the motor-imagery recording has been collected (with the help of our paradigm), we can prepare the data for analysis. EEG is recorded with electrodes placed on the scalp. These measurements contain brain activity, but can also include unwanted signals called artifacts, for example from eye blinks, muscle activity (EMG), or electrical interference (like line noise). Preprocessing helps reduce unwanted noise so that later steps, like feature extraction and training a classifier, can be done more effectively.

In this tutorial, we will use a recording collected for the Cortical Peaks Challenge 2026. The recording is stored as a `.xdf` file. A XDF file can contain several synchronized streams, such as EEG measurements and event markers that indicate when each motor-imagery task occurred. We will first load the file with the `pyxdf` package and inspect the streams and their metadata. This lets us identify the EEG data and its markers before preparing the signal for analysis.

If `pyxdf` is not installed in your Python environment, install it with `pip install pyxdf` before running the code below.

In [78]:
import pyxdf 
import pandas as pd
import mne

file = "data/example.xdf"
streams, header = pyxdf.load_xdf(file)

print(header['info'])
print(streams[0]['info'])
print(streams[1]['info'])
print(streams[2]['info'])

defaultdict(<class 'list'>, {'version': ['1.0']})
defaultdict(<class 'list'>, {'name': ['BrainVision RDA Markers'], 'type': ['Markers'], 'channel_count': ['1'], 'nominal_srate': ['0'], 'channel_format': ['string'], 'source_id': ['RDA 127.0.0.1:51244 Marker'], 'version': ['1.1000000000000001'], 'created_at': ['601433.20920259994'], 'uid': ['dc471b2a-251f-4c50-9d75-e073464f35ad'], 'session_id': ['default'], 'hostname': ['bci108'], 'v4address': [None], 'v4data_port': ['16574'], 'v4service_port': ['16574'], 'v6address': [None], 'v6data_port': ['16574'], 'v6service_port': ['16574'], 'desc': [None], 'stream_id': 2, 'effective_srate': 0, 'segments': [], 'clock_segments': []})
defaultdict(<class 'list'>, {'name': ['BrainVision RDA'], 'type': ['EEG'], 'channel_count': ['35'], 'nominal_srate': ['500'], 'channel_format': ['float32'], 'source_id': ['RDA 127.0.0.1:51244'], 'version': ['1.1000000000000001'], 'created_at': ['601433.20274750004'], 'uid': ['88a5ce1b-f73e-4c34-b023-06e3a7092b0e'], 'sess

The XDF header contains file-level information; in this example it only tells us the XDF format version. The useful recording details are inside each stream's `info` metadata. A stream is one synchronized source of data stored in the XDF file.

This particular file contains three streams:

- `streams[0]` is an empty `BrainVision RDA Markers` stream. It has no recorded events, so we will not use it here.
- `streams[1]` is the `BrainVision RDA` EEG stream. Its `info` describes the recording, including the channel names and sampling rate. Its `time_series` contains the EEG measurements: rows are time samples and columns are channels. The corresponding `time_stamps` tell us when those samples were recorded.
- `streams[2]` is the `paradigm` marker stream. Its `time_series` contains event names such as `left_hand`, `right_hand`, `feet`, and `rest`; its `time_stamps` say when those events occurred.

The EEG stream tells us what the measured signal looked like, while the markers tell us what the participant was asked to imagine and when. We need both: marker timestamps let us locate the relevant EEG around each task cue, then split the continuous signal into trials (also called epochs) and assign each trial its correct class label. Those labeled EEG trials can then be used for feature extraction and classifier training. The stream indices above describe this example file only; other XDF files may list their streams in a different order.

But let's not get ahead of ourselfes. 

TODO: finish writing the markdown and continue the preprocessing 

In [68]:
eeg_idx = next(i for i, s in enumerate(streams) if s["info"]["type"][0] == "EEG")

marker_idx = next(i for i, s in enumerate(streams) if s["info"]["type"][0] == "Markers" and s["info"]["name"][0] == "paradigm")

eeg_stream = streams[eeg_idx]
marker_stream = streams[marker_idx]

all_channel_names = [channel["label"][0] for channel in eeg_stream["info"]["desc"][0]["channels"][0]["channel"]]

eeg_data = pd.DataFrame(eeg_stream["time_series"], columns=all_channel_names)

print("EEG stream index:", eeg_idx)
print("Marker stream index:", marker_idx)
print(eeg_data.shape)
print(marker_stream["time_series"][:])
#next basically tells us where it finds the thing we look for first 
first_pause= next(i for i,s in enumerate(marker_stream['time_series']) if s == ['pause'])
print(f'We find the first pause at position: {first_pause} (start at 0)')
#if we want to sum it up we want to use sum so we see how many pauses were done 
all_pause= sum(i for i,s in enumerate(marker_stream['time_series']) if s == ['pause'])
print(f'sum of all the pauses: {all_pause}')


EEG stream index: 1
Marker stream index: 2
(428900, 35)
[['left_hand'], ['pause'], ['rest'], ['pause'], ['feet'], ['pause'], [' right_hand'], ['pause'], ['feet'], ['pause'], ['rest'], ['pause'], ['rest'], ['pause'], [' right_hand'], ['pause'], ['rest'], ['pause'], ['left_hand'], ['pause'], ['rest'], ['pause'], ['left_hand'], ['pause'], ['rest'], ['pause'], [' right_hand'], ['pause'], [' right_hand'], ['pause'], ['feet'], ['pause'], ['feet'], ['pause'], ['left_hand'], ['pause'], ['rest'], ['pause'], [' right_hand'], ['pause'], ['left_hand'], ['pause'], ['left_hand'], ['pause'], [' right_hand'], ['pause'], ['feet'], ['pause'], ['feet'], ['pause'], ['feet'], ['pause'], [' right_hand'], ['pause'], ['left_hand'], ['pause'], ['feet'], ['pause'], ['rest'], ['pause'], ['left_hand'], ['pause'], ['left_hand'], ['pause'], [' right_hand'], ['pause'], ['rest'], ['pause'], ['feet'], ['pause'], [' right_hand'], ['pause'], ['rest'], ['pause'], ['left_hand'], ['pause'], [' right_hand'], ['pause'], ['fe

In [83]:
channel_meta = streams[stream_idx]['info']["desc"][0]["channels"][0]["channel"]
all_channel_names = []
for elem in channel_meta: 
    all_channel_names.append(elem['label'][0])
all_channel_types = []
for elem in channel_meta:
    channel_type = elem.get('type', ['eeg'])[0].lower()
    channel_name = elem['label'][0].strip().lower()
    if channel_name == 'bip' or channel_type == 'aux':
        all_channel_types.append('misc')
    else:
        all_channel_types.append(channel_type)


# Gather the EEG info and create an MNE RawArray object
eeg_info = streams[stream_idx]['info']
tmp_info = mne.create_info(ch_names=list(eeg_data.columns), ch_types=all_channel_names, sfreq=eeg_info["nominal_srate"][0])
tmp_raw = mne.io.RawArray(eeg_stream.T, tmp_info)

KeyError: "kind must be one of ['grad', 'mag', 'ref_meg', 'eeg', 'seeg', 'dbs', 'ecog', 'eog', 'emg', 'ecg', 'resp', 'bio', 'misc', 'stim', 'exci', 'syst', 'ias', 'gof', 'dipole', 'chpi', 'fnirs_cw_amplitude', 'fnirs_fd_ac_amplitude', 'fnirs_fd_phase', 'fnirs_td_gated_amplitude', 'fnirs_td_moments_intensity', 'fnirs_td_moments_mean', 'fnirs_td_moments_variance', 'fnirs_od', 'hbo', 'hbr', 'csd', 'temperature', 'gsr', 'eyegaze', 'pupil'], not Fp1"

## Step 3: feature extraction

Theorie dann Micha bitte danke hihi 

## Step 4: training a Classifier and benchmarking (??)

Macht Sebastian 